In [1]:
import matplotlib.pyplot as plt
import matplotlib
import scienceplots
from scipy.stats import gaussian_kde
from itertools import islice

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import matplotlib.patches as patches

import numpy as np
import pandas as pd

import os
import shutil

from tqdm import tqdm

os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
print("LaTeX found:", shutil.which('latex'))  # Should return a valid path

# Enable LaTeX
matplotlib.rcParams['text.usetex'] = True

# Set the style
plt.style.use(['science','ieee', 'std-colors'])
plt.rcParams.update({'figure.dpi': '800'})
plt.rcParams.update(plt.rcParamsDefault)


LaTeX found: /usr/bin/latex


In [2]:

from utils.training_utils import read_yaml
from downstream.training_script import main

from glob import glob
from downstream.utils.data_protocol_utils import sanity_check_labels_exist
from downstream.utils import load_data
from downstream.training_script import get_models_pretrained
import buteo as beo

import torch
torch.set_default_device('cuda')

REGIONS = ['denmark-1', 'denmark-2', 'east-africa', 'egypt-1', 'eq-guinea', 'europe', 'ghana-1',
            'isreal-1', 'isreal-2', 'japan', 'nigeria', 'north-america', 'senegal', 'south-america',
            'tanzanipa-1', 'tanzania-2', 'tanzania-3', 'tanzania-4', 'tanzania-5', 'uganda-1']

In [ ]:
import sys
import os
from contextlib import contextmanager

@contextmanager
def suppress_output():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def load_and_crop(file_path, crop_image):
    data = np.load(file_path, mmap_mode='r')
    if (crop_image is not None) and (data.ndim >= 3):
        data = data[:crop_image, :crop_image] if data.ndim == 3 else data[:, :crop_image, :crop_image, :]
    return data

def get_loader(dataset_folder, model_name, downstream_task, input_size, pad_bands, num_classes, n_shot):

    weights, pos_weight, dl_train, dl_test, dl_val, dl_inference= load_data.load_data(
        dataset_folder,
        with_augmentations=False,
        batch_size=16,
        downstream_task=downstream_task,
        model_name=model_name.split('_')[0],
        device="cuda",
        pad_bands=pad_bands,
        crop_images=False, 
        num_classes=num_classes, 
        n=n_shot, 
        weights_dir=downstream_task
    )

    return dl_test

def load_model(args):
    
    with suppress_output():
        model = get_models_pretrained(args.model_name_arch, args.input_channels, args.output_channels, args.input_size, 
                                    args.pretrained_model_path, args.freeze_pretrained)

    checkpoint = torch.load(args.final_model_path)
    
    if 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']

    new_checkpoint = {}
    for k, v in checkpoint.items():
        new_key = k
        if k.startswith('module.'):
            new_key = k[len('module.'):]  # Drop 'module.'
        new_checkpoint[new_key] = v

    # Actually load the weights, but allow for mismatches
    load_result = model.load_state_dict(new_checkpoint, strict=False)

    # Print information about missing and unexpected keys
    if load_result.missing_keys:
        print("Missing keys (parameters in model not found in checkpoint):")
        for key in load_result.missing_keys:
            print(f"  {key}")

    if load_result.unexpected_keys:
        print("Unexpected keys (parameters in checkpoint not found in model):")
        for key in load_result.unexpected_keys:
            print(f"  {key}")

    # if not load_result.missing_keys and not load_result.unexpected_keys:
    #     print("Weights loaded perfectly!")

    return model


In [4]:
def get_testset(folder: str):
    tasks = ['lc', 'lc_classification', 'building', 'roads']

    # 1) gather all the x files up front
    x_test_files = []
    for region in REGIONS:
        pattern = os.path.join(folder, f"{region}*test_s2.npy")
        x_test_files.extend(sorted(glob(pattern)))

    # 2) for each task, re-generate y_paths from the *current* x_test_files,
    #    run the sanity check, and store the filtered results
    y_test_files = {}
    for task in tasks * 3:
        # build the y-filenames for this task, based on whatever x_test_files is now
        y_paths = [x.replace('s2', f'label_{task}') for x in x_test_files]

        x_test_files, kept_y = sanity_check_labels_exist(x_test_files, y_paths)

        # keep only the surviving y’s for this task
        y_test_files[task] = kept_y

    return x_test_files, y_test_files

x_test_files, y_test_files  = get_testset(folder='/Data_phisat2/np_patches_224/')

Showing up to 5 missing files: ['/Data_phisat2/np_patches_224/east-africa_10_test_label_building.npy', '/Data_phisat2/np_patches_224/east-africa_150_test_label_building.npy', '/Data_phisat2/np_patches_224/east-africa_160_test_label_building.npy', '/Data_phisat2/np_patches_224/east-africa_210_test_label_building.npy', '/Data_phisat2/np_patches_224/east-africa_30_test_label_building.npy']
Showing up to 5 missing files: ['/Data_phisat2/np_patches_224/denmark-1_10_test_label_roads.npy', '/Data_phisat2/np_patches_224/denmark-1_20_test_label_roads.npy', '/Data_phisat2/np_patches_224/denmark-1_30_test_label_roads.npy', '/Data_phisat2/np_patches_224/denmark-1_40_test_label_roads.npy', '/Data_phisat2/np_patches_224/denmark-1_50_test_label_roads.npy']


In [5]:
def get_args(model_name, downstream_task):
    args = read_yaml(f"/home/phisat2/phi2FM/downstream/args/phisat2/{model_name}.yml")
    args.freeze_pretrained = False
    args.n_shot = 50
    args.batch_size = 1
    args.downstream_task = downstream_task
    args.output_channels = 1 if 'building' in args.downstream_task or 'roads' in args.downstream_task else 11
    args.model_name_arch = args.model_name + '_classifier' if 'classification' in args.downstream_task else args.model_name

    task_map = {'lc': 'lc', 'lc_classification':'lcc', 'building': 'blg', 'roads':'rds'}
    
    
    if args.model_name == 'GeoAware_mh_pred_core_nano':
        model_name_simple = 'geoaware'
    elif 'phileo_precursor' in args.model_name:
        model_name_simple = 'uniphi'
    elif 'seasonal_contrast' in args.model_name:
        model_name_simple = 'seco'
    else:
        model_name_simple = args.model_name.lower()


    args.final_model_path = f"/home/phisat2/best_models/{model_name_simple}/{task_map[args.downstream_task]}.pt"

    args.only_get_datasets = True
    
    return args

In [9]:
df = pd.read_csv('metrics.csv', index_col=0)
df.dropna(inplace=True)

# 1. compute ranks & aggregate
ranked = df.rank(
    axis=0,
    method='min',
    ascending=[False, False, True, True],
    numeric_only=True
)
ranked['aggregate_rank'] = ranked.sum(axis=1)

# 2. rename those rank-columns so they won’t collide
ranked = ranked.rename(columns={c: f"{c}_rank" for c in df.columns})

# 3. join the metrics
best_overall = ranked.join(df).sort_values('aggregate_rank')

best_overall.head(30)


,lc_rank,lc_classification_rank,building_rank,roads_rank,aggregate_rank,lc,lc_classification,building,roads
11992,1.0,15.0,42.0,53.0,111.0,0.000000,0.818182,0.027071,0.014915
11802,1.0,62.0,7.0,54.0,124.0,0.000000,0.909091,0.018129,0.014928
8221,34.0,62.0,25.0,13.0,134.0,0.000857,0.909091,0.023337,0.011505
14183,95.0,15.0,27.0,17.0,154.0,0.110810,0.818182,0.023799,0.011919
128,55.0,1.0,56.0,43.0,155.0,0.044324,0.727273,0.029605,0.013911
8208,1.0,62.0,65.0,34.0,162.0,0.000000,0.909091,0.031225,0.013002
13702,60.0,15.0,78.0,20.0,173.0,0.050442,0.818182,0.032507,0.012356
8210,1.0,62.0,88.0,23.0,174.0,0.000000,0.909091,0.035312,0.012575
2230,130.0,15.0,24.0,9.0,178.0,0.180624,0.818182,0.023017,0.010437
8233,1.0,138.0,18.0,24.0,181.0,0.000000,1.000000,0.021320,0.012616


In [10]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

TARGET_HEIGHT = 224
TARGET_WIDTH = 224

def process_batch_tiled(model, x_batch, patch_size, target_h, target_w, device, stride=None):
    model.eval() # Ensure model is in eval mode
    B, C, H, W = x_batch.shape

    # Ensure input has the target dimensions before tiling
    if H != target_h or W != target_w:
        # Option 1: Resize - May lose info, but simpler if this is intended behavior sometimes
        print(f"  Warning: Input sample size {H}x{W} differs from target {target_h}x{target_w}. Resizing before tiling.")
        x_batch = F.interpolate(x_batch, size=(target_h, target_w), mode='bilinear', align_corners=False)
        B, C, H, W = x_batch.shape # Update H, W

    if stride is None:
        stride = patch_size # Non-overlapping tiles

    # 1. Padding
    pad_h = (patch_size - H % patch_size) % patch_size
    pad_w = (patch_size - W % patch_size) % patch_size
    padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2)
    x_padded = F.pad(x_batch, padding, mode='reflect')
    H_pad, W_pad = x_padded.shape[-2:]

    # 2. Unfold (Extract Patches)
    patches = F.unfold(x_padded, kernel_size=patch_size, stride=stride)
    num_patches_h = (H_pad - patch_size) // stride + 1
    num_patches_w = (W_pad - patch_size) // stride + 1
    num_patches_total = num_patches_h * num_patches_w
    assert patches.shape[-1] == num_patches_total, "Number of patches mismatch"

    # 3. Reshape Patches
    patches_reshaped = patches.view(B, C, patch_size, patch_size, num_patches_total)
    patches_reshaped = patches_reshaped.permute(0, 4, 1, 2, 3).contiguous()
    patches_final = patches_reshaped.view(B * num_patches_total, C, patch_size, patch_size)

    # 4. Batch Inference
    output_patches = []
    mini_batch_size = 32 # Adjust based on GPU memory
    with torch.no_grad():
        for i in range(0, patches_final.shape[0], mini_batch_size):
             mini_batch = patches_final[i:i+mini_batch_size].to(device)
             output_mini_batch = model(mini_batch)
             output_patches.append(output_mini_batch.cpu())

    output_patches_tensor = torch.cat(output_patches, dim=0)
    
    if len(output_patches_tensor.shape) < 4:
        values, _ = output_patches_tensor.max(dim=0)
        return values.unsqueeze(0)
    
    
    _, C_out, _, _ = output_patches_tensor.shape

    # 5. Reshape for Folding
    output_patches_reshaped = output_patches_tensor.view(B, num_patches_total, C_out, patch_size, patch_size)
    output_patches_reshaped = output_patches_reshaped.permute(0, 2, 3, 4, 1).contiguous()
    output_to_fold = output_patches_reshaped.view(B, C_out * patch_size * patch_size, num_patches_total)

    # 6. Fold (Reconstruct)
    output_padded = F.fold(output_to_fold, output_size=(H_pad, W_pad), kernel_size=patch_size, stride=stride)

    # 7. Normalization
    norm_mask = torch.ones_like(x_padded[:, :1, :, :]) # Use C=1 for efficiency
    norm_mask_patches = F.unfold(norm_mask, kernel_size=patch_size, stride=stride)
    norm_map = F.fold(norm_mask_patches, output_size=(H_pad, W_pad), kernel_size=patch_size, stride=stride)
    norm_map = norm_map.clamp(min=1e-6)
    output_padded_normalized = output_padded / norm_map

    # 8. Crop to Target Size
    crop_top = padding[2]
    crop_left = padding[0]
    final_output = output_padded_normalized[:, :, crop_top:crop_top + H, crop_left:crop_left + W]

    assert final_output.shape[-2:] == (H, W), f"Final output shape spatial mismatch: {final_output.shape} vs {(H,W)}"
    # If the original input was resized, H/W here will be target_h/target_w
    # Ensure the final output matches the dimensions before potential resizing within this function
    assert final_output.shape == (B, C_out, target_h, target_w), "Final output shape mismatch"


    return final_output.to(device)


In [ ]:
out = {}
out_label = {}
# for key in out_label.keys():
#     out_label[key] = {}
#     for task in ['lc', 'lc_classification', 'building', 'roads']:
#         out_label[key][task] = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_list = ['caco', 'dino', 'gassl', 'geoaware', 'moco', 'phisatnet', 'prithvi', 'satmae', 'seco', 'uniphi']
task_list = ['lc', 'lc_classification', 'building', 'roads']

dataset_dir = { 
     "fire": "/Data/fire_dataset/fire_dataset.zarr", 
     "burned_area": "/Data/lpl_burned_area/burned.zarr", 
     "clouds": "/Data/phisatnet_clouds/phisatnet_clouds.zarr", 
     "worldfloods": "/Data/worldfloods/worldfloods.zarr", 
}    
classes = {
     "fire": ['safe', 'fire', 'burnt', 'water'],
     "burned_area": ['Background', 'Burned Area', 'Clouds', 'Waterbodies'],
     "clouds": ['No cloud', 'Cloud', 'value2', 'value3', 'value4'],
     "worldfloods": ['Clouds', 'Land', 'Water'],
}
model_names = {
    "CaCO": "phi2_caco",
    "DINO": "phi2_dino", 
    "GASSL": "phi2_gassl", 
    "GeoAware": "phi2_GeoAware", 
    "MoCo": "phi2_moco", 
    "PhisatNet": "phisatnet_downstream", 
    "Prithvi 1.0": "phi2_prithvi", 
    "SatMAE": "phi2_SatMAE", 
    "SeCo": "phi2_seasonal_contrast", 
    "UniPhi": "phi2_phileo_precursor", 
}
results_dir = {
     
}
# --- Loop through models ---
for model_name in tqdm(model_list, desc="Models"):
    out[model_name] = {}
    for downstream_task in task_list: # Typically just 'building' in this case
        print(f"\nProcessing Model: {model_name}, Task: {downstream_task}")
        args = get_args(model_name=model_name, downstream_task=downstream_task)
        dl_test = get_loader(dataset_dir[args.task], args.model_name, downstream_task, args.input_size, args.pad_bands, len(classes[args.task]), 5000)
        x_sample, y_sample = dl_test.dataset[1000]
        
     #    if args.input_size not in out_label:
     #         out_label[args.input_size] = {}
     #    out_label[args.input_size][downstream_task] = y_sample # Store original label
     #    # Decide how to store the image representation based on your original logic
     #    if args.input_size == 96: # Example condition from your code
     #         out_label[args.input_size]['img'] = x_sample / 10000 # Apply scaling if needed
     #    else:
     #         out_label[args.input_size]['img'] = x_sample
     
     #    out_label[model_name] = x_sample
     
        if model_name == 'phisatnet':
             out_label[downstream_task] = y_sample
             out_label['img'] = x_sample
             
        
        # --- End of sample info storing ---

        model = load_model(args)
        model = model.to(device)
        model.eval()

        # Add batch dimension for the model/tiling function
        x = x_sample.unsqueeze(0) # Shape: [1, C, H, W]
        _, C, H, W = x.shape

        output = None
        with torch.no_grad():
            # Check if input size matches model's expected size
            if H == args.input_size and W == args.input_size:
                # Sizes match, run direct inference
                print(f"  Direct inference (Input: {H}x{W}, Model needs: {args.input_size}x{args.input_size})")
                x = x.to(device)
                print(x.shape)
                output = model(x)
            elif H == TARGET_HEIGHT and W == TARGET_WIDTH:
                 # Sizes mismatch, use tiling (assuming input H/W is the target H/W)
                 print(f"  Tiled inference (Input: {H}x{W}, Model needs: {args.input_size}x{args.input_size})")
                 output = process_batch_tiled(model, x, args.input_size, TARGET_HEIGHT, TARGET_WIDTH, device, stride=args.input_size) # Non-overlapping example
                 # For overlap, try: stride = args.input_size // 2
            else:
                 # Handle case where the loaded sample size H, W is unexpected
                 print(f"  WARNING: Sample size {H}x{W} doesn't match target {TARGET_HEIGHT}x{TARGET_WIDTH} or model input {args.input_size}x{args.input_size}.")
                 # Option: Try resizing to target and then tiling (as done in process_batch_tiled)
                 print(f"  Attempting resize to {TARGET_HEIGHT}x{TARGET_WIDTH} then tiling.")
                 output = process_batch_tiled(model, x, args.input_size, TARGET_HEIGHT, TARGET_WIDTH, device, stride=args.input_size)

            # Store the result for this model and task
            if output is not None:
                 # Output shape should be [1, C_out, TARGET_HEIGHT, TARGET_WIDTH]
                 print(f"  Output generated with shape: {output.shape}")
                 out[model_name][downstream_task] = output.cpu() # Store on CPU
            else:
                 out[model_name][downstream_task] = None # Indicate failure/skip

        # Clean up memory for the next model
        del model, args
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Models:   0%|          | 0/10 [00:00<?, ?it/s]


Processing Model: caco, Task: lc
  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 4, 224, 224])
  Output generated with shape: torch.Size([1, 11, 224, 224])

Processing Model: caco, Task: lc_classification
  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 4, 224, 224])
  Output generated with shape: torch.Size([1, 11])

Processing Model: caco, Task: building
  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 4, 224, 224])
  Output generated with shape: torch.Size([1, 1, 224, 224])

Processing Model: caco, Task: roads


Models:  10%|█         | 1/10 [00:01<00:16,  1.84s/it]

  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 4, 224, 224])
  Output generated with shape: torch.Size([1, 1, 224, 224])

Processing Model: dino, Task: lc
  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 13, 224, 224])
  Output generated with shape: torch.Size([1, 11, 224, 224])

Processing Model: dino, Task: lc_classification
  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 13, 224, 224])
  Output generated with shape: torch.Size([1, 11])

Processing Model: dino, Task: building
  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 13, 224, 224])
  Output generated with shape: torch.Size([1, 1, 224, 224])

Processing Model: dino, Task: roads


Models:  20%|██        | 2/10 [00:04<00:17,  2.23s/it]

...

  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 10, 224, 224])
  Output generated with shape: torch.Size([1, 1, 224, 224])

Processing Model: seco, Task: roads


Lightning automatically upgraded your loaded checkpoint from v1.1.4 to v2.1.3. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../phisat2_foundation/phisat2_fm/pretrained_weights/seco_resnet50_1m.ckpt`
Models:  90%|█████████ | 9/10 [00:33<00:05,  5.37s/it]

  Direct inference (Input: 224x224, Model needs: 224x224)
torch.Size([1, 10, 224, 224])
  Output generated with shape: torch.Size([1, 1, 224, 224])

Processing Model: uniphi, Task: lc
  Tiled inference (Input: 224x224, Model needs: 64x64)
  Output generated with shape: torch.Size([1, 11, 224, 224])

Processing Model: uniphi, Task: lc_classification
  Tiled inference (Input: 224x224, Model needs: 64x64)
  Output generated with shape: torch.Size([1, 11])

Processing Model: uniphi, Task: building
  Tiled inference (Input: 224x224, Model needs: 64x64)
  Output generated with shape: torch.Size([1, 1, 224, 224])

Processing Model: uniphi, Task: roads


Models: 100%|██████████| 10/10 [00:37<00:00,  3.80s/it]

  Tiled inference (Input: 224x224, Model needs: 64x64)
  Output generated with shape: torch.Size([1, 1, 224, 224])


In [33]:
print(out.keys())
print(out['caco'].keys())
print([out['caco'][task].shape for task in out['caco'].keys()])

print('----------------------')

print(out_label.keys())
print([out_label[task].shape for task in out_label.keys()])

dict_keys(['caco', 'dino', 'gassl', 'geoaware', 'moco', 'phisatnet', 'prithvi', 'satmae', 'seco', 'uniphi'])
dict_keys(['lc', 'lc_classification', 'building', 'roads'])
[torch.Size([1, 11, 224, 224]), torch.Size([1, 11]), torch.Size([1, 1, 224, 224]), torch.Size([1, 1, 224, 224])]
----------------------
dict_keys(['lc', 'img', 'lc_classification', 'building', 'roads'])
[torch.Size([1, 224, 224]), torch.Size([8, 224, 224]), torch.Size([11]), torch.Size([1, 224, 224]), torch.Size([1, 224, 224])]


In [34]:
import matplotlib.pyplot as plt
from matplotlib import patches
import matplotlib.lines as mlines
from matplotlib.colors import BoundaryNorm


import matplotlib as mpl
import scienceplots

import numpy as np
import pandas as pd

import os
import shutil

os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
print("LaTeX found:", shutil.which('latex'))  # Should return a valid path

# Enable LaTeX
matplotlib.rcParams['text.usetex'] = True

# Set the style
plt.style.use(['science','ieee', 'std-colors'])
# plt.rcParams.update({'figure.dpi': '800'})
# plt.rcParams.update(plt.rcParamsDefault)


LaTeX found: /usr/bin/latex


In [35]:
# --------------------------------------------------------------
# 0.  Colour map + class names + plotters (Unchanged)
# --------------------------------------------------------------
mpl.rcParams.update({
    'font.size': 28,        # ─── base font size for *all* text ───
})

legend_fontsize = 25

map_code_to_color = {
    0 : (0  , 100,   0),    # Tree cover
    1 : (255, 187,  34),    # Shrubland
    2 : (255, 255,  76),    # Grassland
    3 : (240, 150, 255),    # Cropland
    4 : (250,   0,   0),    # Built-up
    5 : (180, 180, 180),    # Bare / sparse vegetation
    6 : (240, 240, 240),    # Snow and Ice
    7 : (0  , 100, 200),    # Permanent water bodies
    8 : (0  , 150, 160),    # Herbaceous wetland
    9 : (0  , 207, 117),    # Mangroves
    10: (250, 230, 160),    # Moss and lichen
}

# human-readable class names (same order as the codes above)
class_labels = [
    "Tree Cover",
    "Shrubland",
    "Grassland",
    "Cropland",
    "Built-up",
    "Bare/Sparse Vegetation",
    "Snow and Ice",
    "Permanent Water",
    "Herbaceous Wetland",
    "Mangroves",
    "Moss and Lichen",
]

codes     = sorted(map_code_to_color)
lc_colors = [tuple(c/255 for c in map_code_to_color[i]) for i in codes]
lc_cmap   = ListedColormap(lc_colors, name="landcover")

bounds = np.arange(len(lc_colors)+1) - 0.5
norm   = BoundaryNorm(bounds, ncolors=len(lc_colors))


# --------------------------------------------------------------
# 1.  Individual plotting helpers (Unchanged)
# --------------------------------------------------------------

def show_segmentation(ax, tensor):
    """Segmentation map (ground-truth [1,H,W] or argmax predictions [1,C,H,W])."""
    if tensor.ndim == 4 and tensor.shape[1] > 1: # Check if it's prediction tensor (1,C,H,W)
        tensor = tensor.argmax(1) # Get class indices
    tensor = tensor.squeeze().cpu() # Squeeze batch/channel dims -> (H,W)
    ax.imshow(tensor, cmap=lc_cmap, norm=norm, interpolation='nearest')
    ax.axis("off")

def show_multilabel(ax, tensor, thresh=0.5):
    """
    Display an 11-element multilabel / probability vector as a 3×4 grid.
    Input tensor shape: [11] or [1, 11]
    """
    if tensor.ndim > 1:
        tensor = tensor.squeeze() # Ensure tensor is 1D

    # Ensure tensor is on CPU before numpy conversion
    tensor = tensor.cpu()

    needs_sigmoid = (tensor.min() < 0) or (tensor.max() > 1)
    # Apply sigmoid only if values are outside [0, 1] range (likely logits)
    vals = (torch.sigmoid(tensor.float()).numpy()
            if needs_sigmoid else tensor.float().numpy())
    chosen = vals > thresh

    rows, cols = 4, 3
    cell_w, cell_h = 1/cols, 1/rows

    for idx in range(rows * cols):
        r, c = divmod(idx, cols)
        x0, y0 = c*cell_w, 1 - (r+1)*cell_h

        face = lc_colors[idx] if idx < len(class_labels) else 'white'
        rect = patches.Rectangle(
            (x0, y0), cell_w, cell_h,
            facecolor=face, edgecolor="black", linewidth=0.5,
            transform=ax.transAxes
        )
        ax.add_patch(rect)

        if idx == rows*cols - 1:
            ax.text(
                x0 + cell_w/2, y0 + cell_h/2, "n/a",
                ha="center", va="center",
                fontsize=legend_fontsize,
                color="gray", fontstyle="italic",
                transform=ax.transAxes
            )
            continue

        # Check index bounds before accessing 'chosen'
        if idx < len(chosen) and chosen[idx]:
            pad = 0.1 * min(cell_w, cell_h)
            line1 = plt.Line2D(
                [x0+pad, x0+cell_w-pad], [y0+pad, y0+cell_h-pad],
                transform=ax.transAxes, linewidth=2, color="k"
            )
            line2 = plt.Line2D(
                [x0+pad, x0+cell_w-pad], [y0+cell_h-pad, y0+pad],
                transform=ax.transAxes, linewidth=2, color="k"
            )
            ax.add_line(line1)
            ax.add_line(line2)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')

def show_building(ax, tensor):
    """Building-density regression (YlOrRd). Input shape [1,H,W] or [1,1,H,W]"""
    ax.imshow(tensor.squeeze().cpu(), cmap='YlOrRd')
    ax.axis("off")

def show_roads(ax, tensor):
    """Road-density regression (PuBu). Input shape [1,H,W] or [1,1,H,W]"""
    ax.imshow(tensor.squeeze().cpu(), cmap='PuBu')
    ax.axis("off")

def show_image(ax, tensor):
    """RGB composite from a multi-spectral stack. Input shape [C,H,W] or [1,C,H,W]"""
    if tensor.ndim == 4:                      # (1,C,H,W) → (C,H,W)
        tensor = tensor.squeeze(0)

    # Ensure tensor is on CPU before processing
    tensor = tensor.cpu()

    # Assuming RGB are channels 2, 1, 0 based on original code
    # Verify this matches your 'img' tensor's channel order (e.g., Sentinel-2 B4,B3,B2)
    # If Sentinel-2 (B2,B3,B4,...), use indices [3, 2, 1] for RGB (B4=Red, B3=Green, B2=Blue)
    rgb_indices = [2, 1, 0] # <<<### ADJUST THIS if your channel order is different ###>>>

    if tensor.shape[0] >= max(rgb_indices) + 1: # Check if required channels exist
        rgb = tensor[rgb_indices].permute(1, 2, 0).float()   # (H,W,3)
        # Robust quantile calculation for normalization/clamping
        q_low = max(0.0, min(1.0, 0.02))
        q_high = max(0.0, min(1.0, 0.98))
        if rgb.numel() > 0:
            # Calculate quantiles per channel if needed, or across all pixels
            # Using overall quantiles for simplicity here:
            lo = torch.quantile(rgb, q_low)
            hi = torch.quantile(rgb, q_high)
            # Avoid division by zero if hi == lo
            rgb = torch.clamp((rgb - lo) / (hi - lo + 1e-6), 0, 1)
        else:
            rgb = torch.zeros_like(rgb) # Handle empty tensor case
        ax.imshow(rgb)
    elif tensor.shape[0] == 1: # Grayscale fallback if only 1 channel
         img_gray = tensor.squeeze(0).float()
         lo, hi = torch.quantile(img_gray, 0.02), torch.quantile(img_gray, 0.98)
         img_gray = torch.clamp((img_gray - lo) / (hi - lo + 1e-6), 0, 1)
         ax.imshow(img_gray, cmap='gray')
    else: # Fallback for insufficient channels
        ax.text(0.5, 0.5, f"Cannot display RGB\n{tensor.shape[0]} channels",
                ha='center', va='center', transform=ax.transAxes)

    ax.axis("off")


# Map each downstream task to its plotting helper (Unchanged)
plotters = {
    "img"              : show_image,
    "lc"               : show_segmentation,
    "lc_classification": show_multilabel,
    "building"         : show_building,
    "roads"            : show_roads,
}

# Task Labels (Unchanged)
task_labels = {
    "img"               : "Input Image",
    "lc"                : "Land Cover\nSegmentation\n(Pixel-Level)",
    "lc_classification" : "Land Cover\nClassification\n(Image-Level)",
    "building"          : "Building Density\nRegression\n(Pixel-Level)",
    "roads"             : "Road Density\nRegression\n(Pixel-Level)",
}

# Model Labels (Unchanged - assuming these are still relevant)
model_labels = {
    "caco"      : "CaCO",
    "phisatnet" : "$\Phi$satNet",
    "geoaware"  : "GeoAware",
    "dino"      : "DINO",
    "gassl"     : "GASSL",
    "moco"      : "MoCo",
    "prithvi"   : "Prithvi 1.0",
    "satmae"    : "SatMAE",
    "seco"      : "SeCo",
    "uniphi"    : "UniPhi",
}

# Row order (down-stream tasks) (Unchanged)
task_order = ["img", "lc", "lc_classification", "building", "roads"]

In [36]:
def plot_model_comparison(
    out_preds,             # Dictionary: {model_name: {task_name: tensor}}
    out_labels,            # Dictionary: {task_name: tensor} (ground truth + img)
    models_to_plot,        # List of model names (keys in out_preds) to plot
    input_gap_ratio=0.2,   # Width relative to a "normal" column (made slightly smaller)
    label_gap_ratio=0.2,   # Width relative to a "normal" column (made slightly smaller)
    input_col_ratio=1.0,   # Width ratio for the new input image column
    label_col_ratio=1.0,   # Width ratio for the label column
    model_col_ratio=1.0,   # Width ratio for each model prediction column
):
    """
    Plots a grid comparing model predictions against ground truth (LABEL)
    for various tasks. The input image is shown in the first column,
    centered vertically.
    """
    # Define the order of tasks to display as rows
    task_order_display = [
        "lc",
        "lc_classification",
        "building",
        "roads",
    ]
    task_order_display = [t for t in task_order_display if t in out_labels]
    n_rows = len(task_order_display)
    n_models = len(models_to_plot)
    if n_rows == 0:
        print("No tasks found in out_labels to display.")
        return

    # ---- build the list of width-ratios for GridSpec ---------------------------
    # 1 col Input + 1 small gap + 1 col Label + 1 small gap + N cols Models
    width_ratios = (
        [input_col_ratio] +      # Input Image Column
        [input_gap_ratio] +      # Gap Column 1
        [label_col_ratio] +      # Label Column
        [label_gap_ratio] +      # Gap Column 2
        [model_col_ratio] * n_models  # Model Columns
    )
    total_cols = len(width_ratios)  # Should now be 4 + n_models

    # ---- create figure & GridSpec ---------------------------------------------
    figsize_width  = max(3 * np.sum(width_ratios), 15)
    figsize_height = max(3 * n_rows, 8)
    fig = plt.figure(figsize=(figsize_width, figsize_height), constrained_layout=True)
    fig.set_constrained_layout_pads(
        w_pad=0.01,    # horizontal pad between axes, in fraction of axis size
        h_pad=0.01,    # vertical pad
        hspace=0,      # additional vertical space
        wspace=0       # additional horizontal space
    )

    
    gs  = fig.add_gridspec(n_rows, total_cols, width_ratios=width_ratios)


    # ---- Column indices -------------------------------------------------------
    col_input    = 0
    col_gap1     = 1
    col_label    = 2
    col_gap2     = 3
    col_model_0  = 4

    # ---- Input Image Column ---------------------------------------------------
    ax_input = fig.add_subplot(gs[:, col_input])
    if 'img' in out_labels:
        plotters["img"](ax_input, out_labels['img'])
    else:
        ax_input.text(0.5, 0.5, "Input N/A", ha="center", va="center", transform=ax_input.transAxes)
        ax_input.axis("off")
    ax_input.set_title(task_labels.get("img", "Input Image"))

    # ---- Prepare grid of axes (skip gap columns) ------------------------------
    axes_grid = [[None]*total_cols for _ in range(n_rows)]
    for r in range(n_rows):
        # Label column
        axes_grid[r][col_label] = fig.add_subplot(gs[r, col_label])
        axes_grid[r][col_label].axis("off")
        # Gaps are automatically empty
        axes_grid[r][col_gap1] = None
        axes_grid[r][col_gap2] = None
        # Model columns
        for j in range(n_models):
            c = col_model_0 + j
            axes_grid[r][c] = fig.add_subplot(gs[r, c])
            axes_grid[r][c].axis("off")

    # ---- Set column titles ----------------------------------------------------
    if n_rows > 0:
        axes_grid[0][col_label].set_title("Label")
        for j, m in enumerate(models_to_plot):
            axes_grid[0][col_model_0 + j].set_title(model_labels.get(m, m))

    # ---- Fill in each cell ----------------------------------------------------
    for r, task in enumerate(task_order_display):
        # Ground truth
        ax_lbl = axes_grid[r][col_label]
        if task in out_labels:
            plotters[task](ax_lbl, out_labels[task])
        else:
            ax_lbl.text(0.5, 0.5, "Label N/A", ha="center", va="center", transform=ax_lbl.transAxes)

        # Predictions
        for j, m in enumerate(models_to_plot):
            ax_p = axes_grid[r][col_model_0 + j]
            if m in out_preds and task in out_preds[m]:
                plotters[task](ax_p, out_preds[m][task])
            else:
                ax_p.text(0.5, 0.5, "Pred. N/A", ha="center", va="center", transform=ax_p.transAxes)

        # Task annotation
        ax_lbl.annotate(
            task_labels.get(task, task),
            xy=(-0.27/label_col_ratio, 0.5),
            xycoords="axes fraction",
            va="center",
            ha="center",
            rotation="vertical",
        )


    # ---------------- Legend & Colour-bars -------------------------------------
    # Adjust placement relative to the bottom of the figure
    # These might need fine-tuning based on the final figure aspect ratio and content
    y_legend_base = -0.09 # Base vertical position (fraction of figure height from bottom)
    legend_height = 0.07 # Relative height for legends/colorbars

    # Calculate available width for legends/colorbars at the bottom
    # This depends on the figure width and the constrained_layout behavior
    # For simplicity, using relative positions and widths based on figure fraction

    # LC Legend
    x_lc = 0.15 # Start near left edge
    width_lc = 0.67 # Relative width for LC legend
    ax_lc = fig.add_axes([x_lc, y_legend_base, width_lc, legend_height])
    ax_lc.axis("off")
    handles = [patches.Patch(facecolor=lc_colors[i], edgecolor="black") for i in codes]
    ax_lc.legend(
        handles,
        class_labels,
        ncol=min(6, len(class_labels)), # Adjust ncol based on number of labels
        loc="center",
        frameon=True,
        borderpad=0.5,      # Reduced padding
        handlelength=1.0,   # Reduced handle length
        columnspacing=1.0,  # Spacing between columns
        fontsize=legend_fontsize,
    )

    # Colorbars
    gap_cb_1 = -0.02 # Gap between LC legend and first colorbar, and between colorbars
    gap_cb_2 = 0.03 # Gap between LC legend and first colorbar, and between colorbars
    width_cb = (1- width_lc - gap_cb_1 - gap_cb_2 - x_lc) / 2 # Remaining width for colorbars

    x_build = x_lc + width_lc + gap_cb_1
    x_road = x_build + width_cb + gap_cb_2

    # Check if colorbars might overflow and adjust widths if necessary
    if x_road + width_cb > 0.98: # If right edge goes beyond 98% of figure width
       overflow = (x_road + width_cb) - 0.98
       # Reduce widths proportionally (simple approach)
       total_cb_width_area = width_lc + width_cb + width_cb + gap_cb_1 + gap_cb_2
       scale_factor = (total_cb_width_area - overflow) / total_cb_width_area
       width_lc *= scale_factor
       width_cb *= scale_factor
       gap_cb_1 *= scale_factor
       gap_cb_2 *= scale_factor
       # Recalculate positions
       x_build = x_lc + width_lc + gap_cb_1
       x_road = x_build + width_cb + gap_cb_2
       print("Warning: Adjusting legend/colorbar widths to fit figure.")

    y_cb = y_legend_base + (legend_height/2 - 0.015) + 0.02

    # Building density colorbar
    cax_build = fig.add_axes([x_build, y_cb, width_cb, 0.03]) # Centered vertically with legend
    sm_build = mpl.cm.ScalarMappable(cmap="YlOrRd", norm=mpl.colors.Normalize(0, 1))
    sm_build.set_array([])
    cb1 = fig.colorbar(sm_build, cax=cax_build, orientation="horizontal")
    cb1.set_label("Building Density", labelpad=3, fontsize=legend_fontsize)
    cb1.set_ticks([0, 0.5, 1])
    cb1.ax.tick_params(labelsize=legend_fontsize)

    # Road density colorbar
    cax_road = fig.add_axes([x_road, y_cb, width_cb, 0.03]) # Centered vertically with legend
    sm_road = mpl.cm.ScalarMappable(cmap="PuBu", norm=mpl.colors.Normalize(0, 1))
    sm_road.set_array([])
    cb2 = fig.colorbar(sm_road, cax=cax_road, orientation="horizontal")
    cb2.set_label("Road Density", labelpad=3, fontsize=legend_fontsize)
    cb2.set_ticks([0, 0.5, 1])
    cb2.ax.tick_params(labelsize=legend_fontsize)

    plt.show()
    
    return fig


models_to_plot = sorted(out.keys())


fig = plot_model_comparison(
    out_preds=out,
    out_labels=out_label,
    models_to_plot=models_to_plot
)

fig.savefig("model_comparison2.pdf", format="pdf", bbox_inches="tight", dpi=500)